# 4) XGboost CV
- This code go through different candidate weather sets and hyperparatmer tuning for each set by Bayesian optimization with five-fold expanding window.
- Final model gets selected.

# 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import math, pickle
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import seaborn as sns
import glob
import os, sys, gc
from pathlib import Path
import ast
import optuna
import optuna.visualization as vis


In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
from utils import eval_util_module 
import importlib
importlib.reload(eval_util_module)

# 2. Import Data:

## 2-1. Opening training data:

In [ ]:
train = pd.read_parquet('3_output/3_2_temp_train_avg_fit.gzip')

In [ ]:
param = {'random':[40,41,42,43],
         'control':['lon','lat','month_cos','month_sin','lac_dim'],
         
         'clim_only':['tmax', 'tmin', 'tdmean', 'vpdmax', 'vpdmin', 'ppt', 'tmean', 'rh',
       'rh_pm', 'rh_am', 'wetbT', 'thi_max', 'thi_min', 'thi_avg',
       'adjthi', 'ag_tmax', 'ag_tmin', 'ag_tmean', 'ag_tdmean', 'ag_rh_am',
       'ag_rh_pm', 'ag_ppt', 'ag_ssrd_wm-2', 'ag_wind_2m',
       'ag_wetbT', 'ag_thi_max', 'ag_thi_min', 'ag_adjthi'],
         
         'clim_only1': ['tmax', 'tmin', 'tdmean', 'vpdmax', 'vpdmin', 'ppt',
       'rh_pm', 'rh_am', 'wetbT',  'thi_max', 'thi_min',
       'adjthi', 'ag_tmax', 'ag_tmin', 'ag_tdmean', 'ag_rh_am',
       'ag_rh_pm', 'ag_ppt', 'ag_ssrd_wm-2', 'ag_wind_2m',
       'ag_wetbT',  'ag_thi_max', 'ag_thi_min', 'ag_adjthi'],
        }

## weighting for data numbers across splits
data_weight = [17276370,20792868,24432506,25761018,23290832]

In [ ]:
expanding_splits = [(2006,2009),
                   (2009,2012),
                   (2012,2015),
                   (2015,2018),
                   (2018,2021)]


## 2-2. candidate weather sets:
- We did a lot of testing for different combinations which are not all included here
    - Use rh from agera5 and vpd from prism after testing.

In [ ]:
feat_list = {}
feat_list.update({
    ####################### New test ###########################################
    #### heat index and individuals
    'thimax':['thi_max'],
    'adjthi': ['adjthi'],
    'thimax_indv':['tmax','rh_pm'],
    'thi_indv':['tmean','rh'],
    'adjthi_indv':['tmean','rh','ag_ssrd_wm-2','ag_wind_2m'],
    'adjthimax_indv': ['tmax','rh_pm','ag_wind_2m','ag_ssrd_wm-2'],
    'basic3':['tmean','rh','tmax_ssrd','ag_wind_2m'],
    
    ## test vpdmin vs. ag_rh (same group): interchangeablei
    'sp1_night_noppt':['tmin','tmax_ssrd','vpdmin','ag_wind_2m'],
    'sp1_day_noppt':['tmin','tmax_ssrd','vpdmax','ag_wind_2m'],
    'sp1_day':['tmin','tmax_ssrd','vpdmax','ag_wind_2m','ppt'],
    'sp1_night':['tmin','tmax_ssrd','vpdmin','ag_wind_2m','ppt'],
    'sp1_day_rhpm':['tmin','tmax_ssrd','rh_pm','ppt','ag_wind_2m'], #prism rh was better than agera5
    'sp1_day_rhpm_noppt':['tmin','tmax_ssrd','rh_pm','ag_wind_2m'],
    'sp1_night_rham_noppt':['tmin','tmax_ssrd','rh_am','ag_wind_2m'],
    'sp1_avg_rh_noppt':['tmin','tmax_ssrd','rh','ag_wind_2m'],
    'sp2_rh_noppt':['tmin','tmax_ssrd','rh_am','rh_pm','ag_wind_2m'],
    'sp2_rh_ag_noppt':['tmin','tmax_ssrd','ag_rh_am','ag_rh_pm','ag_wind_2m'],
    'sp1_wetbt_noppt':['wetbT','tmin','tmax_ssrd','rh_am','ag_wind_2m'],
    'sp1_wetbt_short':['wetbT','tmax_ssrd','ag_wind_2m'],
    
    ### Knowledge_based
    'basic1':['tmax','tmin','tdmean','ag_wind_2m','ag_ssrd_wm-2','ppt'], #lowest 1
    'basic2':['tmin','tmean_ssrd','rh_pm','ag_wind_2m'],
})


## 2-3) Baseline output

In [ ]:
base_output = pd.read_csv('3_output/3_1_baseline_error_score.csv', index_col=0)
base_output.loc[base_output['category'] == 'val', 'category'] = 'validation'

# 3. Hyperparameter Tuning:
- Hyperparameters were optimized individually by bayesian optimization with five-folding expanding widnow cross-validation.
- To use the synthetic data, we changed the setting varialbes below (hyperparameter ranges, n_trials, warm_num, cand, and patience). These numbers should be changed depending on your data.

In [ ]:
def early_stopping_callback(patience, min_trials):
    best_score = float("inf")
    no_improve_count = 0
    tracking_started = False

    def callback(study, trial):
        nonlocal best_score, no_improve_count, tracking_started

        if len(study.trials) < min_trials:
            return  # Don't track early stopping yet

        if not tracking_started:
            # Reset after min_trials
            best_score = trial.value
            no_improve_count = 0
            tracking_started = True
            return

        if trial.value < best_score:
            best_score = trial.value
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= patience:
            print(f"[Early stop] No improvement in {patience} trials after {min_trials} → Stopping.")
            study.stop()

    return callback


In [ ]:
name_list = list(feat_list.keys())
file_name = 'cv_results'
len(name_list)

In [ ]:
#### creating empty output df:
df_results = pd.DataFrame()
cv_results = []

for feat in name_list:

    feature = param['control'] + feat_list[feat]
    print(feat, '-----------------------------------------')
    col_len = len(feature)

    def objective(trial):
        
        # Define hyperparameter search space
        ## Dividing search space based on number of features to reduce number of trials
        ###############################################################################
        ######### Setting: these ranges should be changed depending on your data ######
        ###############################################################################
        
        if col_len <= 7:
            params = {
                    'objective':'reg:squarederror',
                    'n_jobs':-1,
                   'seed': param['random'][0],
                    "max_depth": trial.suggest_int("max_depth", 3, 7, step=1),
                    "learning_rate":trial.suggest_float("learning_rate", 0.05, 0.1, step=0.05),
                    "min_child_weight": trial.suggest_int("min_child_weight", 5, 20, step=5),
                    "subsample": trial.suggest_float("subsample", 0.8, 1, step=0.1),
                    "colsample_bytree": 1,
                    }
            num_boost_round = trial.suggest_int("num_boost_round", 20, 80, step=10)
        else:
            params = {
            'objective':'reg:squarederror',
                    'n_jobs':-1,
                    'seed': param['random'][0],
                    "max_depth": trial.suggest_int("max_depth", 5, 10, step=1),
                    "learning_rate":trial.suggest_float("learning_rate", 0.05, 0.1, step=0.05),
                    "min_child_weight": trial.suggest_int("min_child_weight", 5, 20, step=5),
                    "subsample": trial.suggest_float("subsample", 0.8, 1, step=0.1),
                    "colsample_bytree": 1,
                    }
            num_boost_round = trial.suggest_int("num_boost_round", 20, 100, step=10)

        # Store cross-validation scores manually
        cv_scores = []

        ## Loop through your custom_cv (expanding window + different targets)
        for idx, (train_end, val_end) in enumerate(expanding_splits): ## fortesting

            print('Fold :', idx+1 ,'--------------------------------','train_end :', train_end, 'val_end :', val_end)
            sub_cols =[]
            
            # 1. Slice the data FIRST (Keep them as Pandas DataFrames)
            train_slice = train.loc[(train['adj_year'] <= train_end)]
            val_slice = train.loc[(train['adj_year'] > train_end) & (train['adj_year'] <= val_end)]

            # 2. CHECK: Are they empty?
            if len(train_slice) == 0 or len(val_slice) == 0:
                print(f"Skipping Fold {idx+1}: Not enough data (Train: {len(train_slice)}, Val: {len(val_slice)})")
                continue # Skip to the next fold iteration immediately

            # ... Setup targets ...
            sub_target = f'herd_milk_resid_{idx + 1}'
            sub_cols = list(feature)

            # 3. Convert to DMatrix (Now we know it's safe)
            train_df = xgb.DMatrix(train_slice[sub_cols], label=train_slice[sub_target])
            val_df = xgb.DMatrix(val_slice[sub_cols], label=val_slice[sub_target])
            
           
            print('training model')
            model = xgb.train(
                params=params,
                dtrain=train_df,  # Training data
                num_boost_round=num_boost_round,
                evals=[(train_df, "train"), (val_df, "validation")],  # Eval data
                early_stopping_rounds = 20,
                verbose_eval=False  # No verbose output
                )
            del train_df
            gc.collect()

            # Fit model and predict
            pval = model.predict(val_df)
            del val_df
            gc.collect()

            # Compute composite metric 
            val_target = train.loc[(train['adj_year'] > train_end) & (train['adj_year']<=val_end)][sub_target]
            loss = mean_squared_error(val_target, pval)
            print(idx+1, np.round(loss,6))

            cv_scores.append(loss)

            del model, val_target, pval
            gc.collect()

        # Return average RMSE across all custom folds
        return np.round(np.average(cv_scores, weights=data_weight),6)


    #########################################################################################
    ################## Setting ##############################################################
    #########################################################################################
    ## Change if you need ###################################################################
    n_trials = 30 
    cand = 5  
    warm_num = 5 
    patience = 3

    # Create study #no pruning is required for me:
    study = optuna.create_study(
            direction="minimize",
            study_name=feat,
            sampler=optuna.samplers.TPESampler(seed=param['random'][0], n_startup_trials=warm_num,
                n_ei_candidates=cand),
            # storage=f"sqlite:///3_output/bayes_study/{feat}.db",
            load_if_exists=True,
            )

    # Run optimization: 
    '''
    try 30 experiments with random search for the first 5 trials,compare 5 internal samples before picking the next trial
    early stop if there is no improvement for the last 5 trials 
    '''
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[early_stopping_callback(patience=patience,min_trials=warm_num)])


    cv_results.append({
        "features": feat,
        "best_params": study.best_params,
        "best_rmse": study.best_value
    })
    ## intermediate output:
    pd.DataFrame(cv_results).to_csv(f'3_output/4_1_tuning_{file_name}.csv')


In [ ]:
cv_results = pd.DataFrame(cv_results)

# 4. Feature Selection:

## 4-1. Training Set CV scores:
- Using the tunned hyperparameters, fit the model across five-fold expnading windows

In [ ]:
cv_output = pd.DataFrame()

for idx, (train_end, val_end) in enumerate(expanding_splits):
    print('Fold :', idx+1 ,'--------------------------------','train_end :', train_end, 'val_end :', val_end)

    ## Since each fold has a different arget name, select it:
    sub_target = f'herd_milk_resid_{idx + 1}'

    for idy, feat in enumerate(name_list):
        
        ## features:
        sub_cols = param['control'] + feat_list[feat]
        
        print(sub_cols,' -----------------------------------')
        if len(sub_cols) <= 7:
            model_param = {
            'seed': param['random'][0],
            'objective': 'reg:squarederror',
            'max_depth':6,
            'learning_rate': 0.1,
            'subsample':1.0,
            'colsample_bytree': 1,
            'min_child_weight': 15,
            'n_jobs': -1, 
            }
            num_boost_round = 60
        else:
            model_param  = {
            'seed': param['random'][0],
            'objective': 'reg:squarederror',
            'max_depth':6,
            'learning_rate': 0.1,
            'subsample':1.0,
            'colsample_bytree': 1,
            'min_child_weight': 15,
            'n_jobs': -1, 
            }
            num_boost_round = 80
            
        print('dmatrix format')
        # 1. Slice the data FIRST (Keep them as Pandas DataFrames)
        train_slice = train.loc[(train['adj_year'] <= train_end)]
        val_slice = train.loc[(train['adj_year'] > train_end) & (train['adj_year'] <= val_end)]

        # 2. CHECK: Are they empty?
        if len(train_slice) == 0 or len(val_slice) == 0:
            print(f"Skipping Fold {idx+1}: Not enough data (Train: {len(train_slice)}, Val: {len(val_slice)})")
            continue # Skip to the next fold iteration immediately
        
        # 3. Convert to DMatrix
        train_df = xgb.DMatrix(train_slice[sub_cols], label=train_slice[sub_target])
        val_df = xgb.DMatrix(val_slice[sub_cols], label=val_slice[sub_target])
    
        print('training model')
        model = xgb.train(
                params=model_param,
                dtrain=train_df,  # Training data
                num_boost_round=num_boost_round,
                evals=[(train_df, "train"), (val_df, "validation")],  # Eval data
                early_stopping_rounds = 20,
                verbose_eval=False  # No verbose output
                )

        ## 5. predict:
        ptrain = model.predict(train_df)
        pval = model.predict(val_df)
        del train_df, val_df
        gc.collect()

        ## output:
        train_metrics = eval_util_module.evaluate(train_slice[sub_target], ptrain)
        val_metrics   = eval_util_module.evaluate(val_slice[sub_target],   pval)
        
        
        # 6. Format Results
        # Create a temporary list of dicts to handle metadata cleanly
        results = []

        # Process Train
        train_metrics.update({'category': 'train', 'fold': idx + 1, 'feature': feat})
        results.append(train_metrics)

        # Process Validation
        val_metrics.update({'category': 'validation', 'fold': idx + 1, 'feature': feat})
        results.append(val_metrics)

        # 4. Save Output
        print('Saving...')
        new_rows = pd.DataFrame(results)
        cv_output = pd.concat([cv_output, new_rows], axis=0, ignore_index=True)
        
        del ptrain, pval, new_rows, results


# Save to CSV
cv_output.to_csv('3_output/4_3_train_cv_performance.csv')
        

# 4-2. Test set scores: 
- This is a hold-out set.

In [ ]:
test = pd.read_parquet('3_output/3_2_temp_test_avg_fit.gzip')

In [ ]:
test_output = pd.DataFrame()

for idy, feat in enumerate(name_list):

    ## features:
    sub_cols = param['control'] + feat_list[feat]
    sub_target = 'herd_milk_resid_train'

    print(sub_cols,' -----------------------------------')
    if len(sub_cols) <= 7:
        model_param = {
        'seed': param['random'][0],
        'objective': 'reg:squarederror',
        'max_depth':6,
        'learning_rate': 0.1,
        'subsample':1.0,
        'colsample_bytree': 1,
        'min_child_weight': 15,
        'n_jobs': -1, 
        }
        num_boost_round = 60
    else:
        model_param  = {
        'seed': param['random'][0],
        'objective': 'reg:squarederror',
        'max_depth':6,
        'learning_rate': 0.1,
        'subsample':1.0,
        'colsample_bytree': 1,
        'min_child_weight': 15,
        'n_jobs': -1, 
        }
        num_boost_round = 80
    print('dmatrix format')


    # 1. Convert to DMatrix (Now we know it's safe)
    train_df = xgb.DMatrix(train[sub_cols], label=train[sub_target])
    test_df = xgb.DMatrix(test[sub_cols], label=test[sub_target])

    print('training model')
    model = xgb.train(
            params=model_param,
            dtrain=train_df,  # Training data
            num_boost_round=num_boost_round,
            evals=[(train_df, "train"), (test_df, "test")],  # Eval data
            early_stopping_rounds = 10,
            verbose_eval=False  # No verbose output
            )

    ## 5. predict:
    ptrain = model.predict(train_df)
    ptest = model.predict(test_df)
    del train_df, test_df
    gc.collect()

    ## output:
    train_metrics = eval_util_module.evaluate(train[sub_target], ptrain)
    test_metrics   = eval_util_module.evaluate(test[sub_target],  ptest)


    # 6. Format Results
    # Create a temporary list of dicts to handle metadata cleanly
    results = []

    # Process Train
    train_metrics.update({'category': 'train', 'fold': np.nan, 'feature': feat})
    results.append(train_metrics)

    # Process Validation
    test_metrics.update({'category': 'test', 'fold': np.nan, 'feature': feat})
    results.append(test_metrics)

    # 4. Save Output
    print('Saving...')
    new_rows = pd.DataFrame(results)
    test_output = pd.concat([test_output, new_rows], axis=0, ignore_index=True)

    del ptrain, ptest, new_rows, results


# Save to CSV
test_output.to_csv('3_output/4_4_test_cv_performance.csv')

# 5. Final Score
- As the performance of the holdout set is important, we average cv validation performance then average it with the holdout set's performance.

In [ ]:
def avg_cv(temp):
    temp_summary = pd.DataFrame()
    
    for col in ['mae','mse','high20%_mse','low20%_mse']:
        #new_var = f'w_'+col
        temp_summary = pd.concat([
            temp_summary,
            temp.loc[temp['category'] == 'validation']
            .groupby(['feature'])
            .apply(lambda x: pd.Series({
                f'{col}': np.average(x[col], weights=data_weight),
                # f'{col}_std':  np.sqrt(
                #     np.average((x[col] - np.average(x[col], weights=n_list))**2,
                #                weights=n_list))
            }))
        ], axis=1)

    temp_summary['rmse'] = np.sqrt(temp_summary['mse'])
    return temp_summary


def avg_scores(error, cv_score, test_score):
    return 0.5*cv_score[error] + 0.5*test_score[error]

name_map ={'sp1_night_rham_noppt':'NightRH','sp1_night_noppt':'NightVPD','sp1_day_noppt':'DayVPD',
            'sp2_rh_noppt':'All_day_RH','sp1_night':'NightVPD_PPT','sp1_wetbt_short':'Wetbt_Solar',
            'sp1_day_rhpm_noppt':'DayRH','sp1_wetbt_noppt':'Wetbt_Night_Solar','sp1_avg_rh_noppt':'Night_RHavg',
            'sp2_rh_ag_noppt':'All_day_RH_agera5','sp1_day':'DayVPD_PPT','sp1_day_rhpm':'DayRH_PPT',
           'thimax':'THImax','thi_indv':'THI_Indiv','adjthi':'AdjTHI','adjthimax_indv':'AdjTHImax_Indiv',
           'baseline':'Baseline','basic1':'Basic1', 'basic2':'Basic2', 'basic3':'Basic3', 'thimax_indv':'THImax_Indiv'
                               }

In [ ]:
## CV output averages:
summary_base = avg_cv(base_output)
summary_cv = avg_cv(cv_output)
summary_cv = pd.concat([summary_cv, summary_base], axis=0)

In [ ]:
## Test output:
summary_test = pd.concat([base_output.loc[base_output['category'] == 'test'][['feature','category','mse','mae']].set_index('feature'), 
                          test_output.loc[test_output['category'] == 'test'][['feature','category','mse','mae']].set_index('feature')], axis=0)
summary_test['rmse'] = np.sqrt(summary_test['mse'])


In [ ]:
## Combining (SI table 1):
avg_rmse = avg_scores('rmse', summary_cv, summary_test)
avg_mae = avg_scores('mae', summary_cv, summary_test)
final_loss = pd.concat([avg_rmse.rename('avg_rmse'), avg_mae.rename('avg_mae')], axis=1).reset_index()
final_loss.sort_values(by='avg_mae')
final_loss['final'] = final_loss['avg_rmse'] * 0.5 + final_loss['avg_mae'] * 0.5

final_loss['feature_name'] = final_loss['feature'].map(name_map)
final_loss.sort_values(by=['final'])

In [ ]:
final_loss.to_csv('3_output/4_5_final_performance.csv', index=False)